# 机器学习中的距离：教学计算与配图

面向理解坐标、绝对值、平方和基本回归的初学者。全部数据为自设教学数据。

按顺序运行：三距离 → 单位换算 → 余弦 → SVM 垂足 → 正则化约束 → 练习 → 重绘全部图片。

请从项目根目录启动 Notebook；不要混用旧文件 `ml_distance_codes.ipynb`。依赖见 `requirements.txt`，源代码见 `scripts/build_figures.py`。

In [1]:
from pathlib import Path
import json
import platform
import numpy as np
import scipy
import sklearn
from sklearn.svm import SVC

# 工作目录应含 scripts/；若从 Notebook 所在目录启动，无需修改。
root = Path.cwd()
assert (root / 'scripts' / 'build_figures.py').exists(), '请从项目根目录运行'
print('Python:', platform.python_version())
print('NumPy:', np.__version__, 'SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)

Python: 3.12.16
NumPy: 2.5.4 SciPy: 1.18.1
scikit-learn: 1.9.1


## 1. 同一对点，三种距离

从 A=(1,1) 到 B=(5,4)，差向量为 (4,3)。预测三个结果，再运行。`ord` 在一维数组上指定向量范数；这里不讨论矩阵范数。

In [2]:
a = np.array([1.0, 1.0])
b = np.array([5.0, 4.0])
delta = b - a
distances = {str(p): float(np.linalg.norm(delta, ord=p))
             for p in (1, 2, np.inf)}
print('L1、L2、L∞:', distances)
assert np.allclose(list(distances.values()), [7.0, 5.0, 4.0])

# 平方欧氏距离不满足一般的三角不等式。
print('平方距离反例:', (2-0)**2, '>', (1-0)**2 + (2-1)**2)

L1、L2、L∞: {'1': 7.0, '2': 5.0, 'inf': 4.0}
平方距离反例: 4 > 2


## 2. 只换单位，邻居也可能改变

三行依次为 Q、A、B，列为收入(万元)、员工人数。把收入换成元之后，哪家公司离 Q 更近？这里展示原始距离的尺度问题，不把这三行当作完整训练数据，也不据此评价模型泛化。

In [3]:
firms = np.array([[100., 20.], [101., 23.], [102., 21.]])
raw = np.linalg.norm(firms[1:] - firms[0], axis=1)
converted = firms.copy()
converted[:, 0] *= 10000.0
after = np.linalg.norm(converted[1:] - converted[0], axis=1)
print('收入万元，Q到[A,B]:', np.round(raw, 6))
print('收入元，  Q到[A,B]:', np.round(after, 6))
assert int(raw.argmin()) == 1 and int(after.argmin()) == 0

收入万元，Q到[A,B]: [3.162278 2.236068]
收入元，  Q到[A,B]: [10000.00045  20000.000025]


## 3. 余弦比较方向

公式只适用于非零向量。正比例缩放使方向不变；正交时相似度为0。对已归一化的非零向量，还可以验证平方欧氏距离与余弦的关系。

In [4]:
def cosine(x, y):
    nx, ny = np.linalg.norm(x), np.linalg.norm(y)
    if nx == 0 or ny == 0:
        raise ValueError('几何余弦公式要求两个向量非零')
    return float(np.dot(x, y) / (nx * ny))

x = np.array([1., 2.])
y = 3.0 * x
z = np.array([2., -1.])
print('正比例缩放:', round(cosine(x, y), 6))
print('正交:', round(cosine(x, z), 6))
assert np.isclose(cosine(x, y), 1.0)
assert np.isclose(cosine(x, z), 0.0)
xu, zu = x / np.linalg.norm(x), z / np.linalg.norm(z)
assert np.isclose(np.linalg.norm(xu-zu)**2, 2*(1-cosine(x, z)))

正比例缩放: 1.0
正交: 0.0


## 4. SVM：从拟合结果算出垂足

负类点的坐标和为2，正类为4。使用线性 SVC 的大 C 求解；本教学样本得到的解满足硬间隔条件。H 为选定点在决策边界上的垂足。检查 H 落在边界上、点到 H 的连线与边界方向垂直。

In [5]:
X = np.array([[0., 2.], [1., 1.], [2., 0.],
              [1., 3.], [2., 2.], [3., 1.]])
labels = np.array([-1, -1, -1, 1, 1, 1])
model = SVC(kernel='linear', C=1e6, tol=1e-10).fit(X, labels)
w, intercept = model.coef_[0], float(model.intercept_[0])
point = np.array([2., 2.])
foot = point - (np.dot(w, point) + intercept) * w / np.dot(w, w)
tangent = np.array([-w[1], w[0]])
margin = labels * model.decision_function(X)
print('w, b:', np.round(w, 9), round(intercept, 9))
print('垂足:', np.round(foot, 9))
print('单侧间隔:', 1 / np.linalg.norm(w))
print('边界残差:', np.dot(w, foot) + intercept)
print('垂直性内积:', np.dot(point-foot, tangent))
assert np.allclose(w, [1., 1.], atol=1e-7)
assert np.isclose(intercept, -3., atol=1e-7)
assert np.all(margin >= 1-1e-7)
assert np.isclose(np.dot(w, foot)+intercept, 0., atol=1e-7)
assert np.isclose(np.dot(point-foot, tangent), 0., atol=1e-7)

w, b: [1. 1.] -3.0
垂足: [1.5 1.5]
单侧间隔: 0.7071067811865475
边界残差: 0.0
垂直性内积: 0.0


## 5. 正则化：同一个损失，不同约束

用 Q(β)=||β−c||₂²，c=(2,0.5)，约束半径1。对这一特别构造的例子，L1 投影为 (1,0)，L2 投影为 c/||c||₂。下面通过通用 L1 球投影计算前者，不手工指定图上的点。

In [6]:
def project_l1_ball(v, radius=1.0):
    # 通过软阈值把向量投影到 L1 球；已可行时原样返回。
    if radius <= 0:
        raise ValueError('本例要求半径为正')
    if np.linalg.norm(v, 1) <= radius:
        return v.copy()
    ordered = np.sort(np.abs(v))[::-1]
    cumulative = np.cumsum(ordered)
    indexes = np.arange(1, len(v)+1)
    rho = np.flatnonzero(ordered > (cumulative-radius)/indexes)[-1]
    threshold = (cumulative[rho]-radius)/(rho+1)
    return np.sign(v) * np.maximum(np.abs(v)-threshold, 0.0)

center = np.array([2., 0.5])
lasso = project_l1_ball(center)
ridge = center / max(1.0, np.linalg.norm(center))
print('L1约束解:', lasso)
print('L2约束解:', np.round(ridge, 6))
assert np.allclose(lasso, [1., 0.])
assert np.allclose(ridge, [4/np.sqrt(17), 1/np.sqrt(17)])

L1约束解: [1. 0.]
L2约束解: [0.970143 0.242536]


## 6. 练习：Lasso 一定产生零系数吗？

把无约束最优点改为 c=(0.8,0.8)，约束半径仍为1。先判断 L1 约束解会不会落在尖角，再用下方代码核对。这个练习用于说明图8是一个例子，不是所有目标函数都必然出现零系数。

In [7]:
exercise_center = np.array([0.8, 0.8])
exercise_solution = project_l1_ball(exercise_center)
print('练习解:', exercise_solution)
assert np.allclose(exercise_solution, [0.5, 0.5])
print('两个系数都非零。L1约束不保证每个例子都产生零系数。')

练习解: [0.5 0.5]
两个系数都非零。L1约束不保证每个例子都产生零系数。


## 7. 从同一源代码重绘全部图片

这一单元运行项目脚本，重新计算 KNN、K-Means、SVM 和约束最优点，保存8张 PNG及同源 SVG，并将检查结果写入 `results/figure-checks.json`。图片输出使用脚本中固定的本轮文件名；新一轮正式返修应更换时间戳。

这是教学几何核验，没有训练/测试预测性能实验，不把训练点全部分对解释为样本外效果。

In [8]:
import runpy

runpy.run_path(str(root / 'scripts' / 'build_figures.py'), run_name='__main__')
evidence_path = root / 'results' / 'figure-checks.json'
evidence = json.loads(evidence_path.read_text(encoding='utf-8'))
print('图形数值证据已保存:', evidence_path.name)
print('本轮正式PNG图形数量:', len(evidence['figures']))

{
  "status": "passed",
  "figures": 8,
  "warnings": [],
  "checks": {
    "three_distances": {
      "A": [
        1.0,
        1.0
      ],
      "B": [
        5.0,
        4.0
      ],
      "delta": [
        4.0,
        3.0
      ],
      "l1": 7.0,
      "l2": 5.0,
      "linf": 4.0,
      "chebyshev_rendering": "标注最大位移分量，不把任意曲线当作路径长度"
    },
    "unit_balls": {
      "l1_vertices": [
        [
          1,
          0
        ],
        [
          0,
          1
        ],
        [
          -1,
          0
        ],
        [
          0,
          -1
        ]
      ],
      "linf_vertices": [
        [
          -1,
          -1
        ],
        [
          1,
          -1
        ],
        [
          1,
          1
        ],
        [
          -1,
          1
        ]
      ],
      "interpretation": "彩色边界满足范数等于 1；浅色区域满足范数小于等于 1。"
    },
    "cosine": {
      "x": [
        3.0,
        0.0
      ],
      "y": [
        2.0,
        2.0
      ],
      "cos_x_y"

## 下一步

先改变一个条件再观察结果：修改特征单位、邻居数量或约束半径。完整正文和运行说明见项目 `README.md`。本文不展开核方法、度量学习或正式预测评价；学习这些内容时，继续区分度量对象、拟合目标和样本外验证。